# RSNA timing test on Kaggle (Phase 0, DEC-17)

Measures whether Vision B can use the full RSNA data on Kaggle. **It trains nothing and saves no images.** It reads small samples, times each step, then estimates the full run. Expect about 10–20 minutes.

**Before you start**
1. Accept the rules on the [competition page](https://www.kaggle.com/c/rsna-intracranial-hemorrhage-detection/).
2. Upload this notebook to Kaggle (Create → New Notebook, then File → Import Notebook).
3. Add the competition data with Add Input, and set the accelerator to **GPU T4**. Internet can stay off.
4. Choose Run All. Send back the summary from the last cell, or `rsna_timing_report.json` from the output folder.

The report holds counts and timings only, no patient IDs or images. The model starts from random weights and data, so its loss means nothing; only its speed is used.

## 1. Settings

In [ ]:
import json, os, platform, random, shutil, time
from collections import Counter

import cv2, numpy as np, pandas as pd, pydicom, torch, torchvision
from torch.utils.data import DataLoader, Dataset

pydicom.config.settings.reading_validation_mode = pydicom.config.IGNORE  # RSNA's IDs are not valid DICOM UIDs

DATA_ROOT = '/kaggle/input'
WORK_DIR = '/kaggle/working'
SEED = 2026
HEADER_SAMPLE = 2000
PIXEL_SAMPLE = 500
LOADER_SAMPLE = 1500
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() or 2
TRAIN_STEPS = 60
WARMUP_STEPS = 10
IMAGE_SIZES = [256, 384]
MODELS = ['resnet50', 'efficientnet_b3']
SESSION_HOURS = 12  # unconfirmed Kaggle limit: check your account's quota page
WEEKLY_GPU_HOURS = 30  # unconfirmed Kaggle limit
OUTPUT_LIMIT_GB = 20  # unconfirmed Kaggle limit

WINDOWS = [(40, 80), (80, 200), (600, 2800)]  # brain, subdural, bone (plan §5.2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
report = {'seed': SEED, 'device': torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu', 'cpu_count': os.cpu_count(), 'python': platform.python_version(), 'pydicom': pydicom.__version__, 'torch': torch.__version__}
print(report)
if device == 'cpu':
    print('WARNING: no GPU found. Set the accelerator to GPU T4, or the training speeds will not be meaningful.')

## 2. Find the data and read the labels

In [ ]:
csv_hits = []
for folder, subfolders, files in os.walk(DATA_ROOT):
    subfolders[:] = [d for d in subfolders if not d.startswith('stage_2_')]  # never list the image folders
    if 'stage_2_train.csv' in files:
        csv_hits.append(os.path.join(folder, 'stage_2_train.csv'))
if not csv_hits:
    raise FileNotFoundError(f'stage_2_train.csv not found under {DATA_ROOT}. Add the competition data with Add Input.')
data_dir = os.path.dirname(csv_hits[0])
image_dir = os.path.join(data_dir, 'stage_2_train')

labels = pd.read_csv(csv_hits[0])
labels = labels[labels['ID'].str.endswith('_any')].drop_duplicates('ID')
labels['image_id'] = labels['ID'].str[:-len('_any')]
n_slices = len(labels)
report['labelled_slices'] = n_slices
report['any_bleed_slice_rate'] = round(float(labels['Label'].mean()), 4)

rng = random.Random(SEED)
sample = labels.sample(n=min(n_slices, max(HEADER_SAMPLE, PIXEL_SAMPLE, LOADER_SAMPLE)), random_state=SEED)
paths = [os.path.join(image_dir, f'{i}.dcm') for i in sample['image_id']]
targets = sample['Label'].astype('float32').tolist()
report['sample_files_missing'] = sum(not os.path.exists(p) for p in paths)
print(f"{n_slices:,} labelled slices, {report['any_bleed_slice_rate']:.1%} with a bleed. Missing in sample: {report['sample_files_missing']}.")

## 3. Read DICOM headers
Checks that slices can be grouped by patient and scan, and measures how long a full header scan would take.

In [ ]:
header_paths = paths[:HEADER_SAMPLE]
start, thickness, failed = time.perf_counter(), Counter(), 0
present = Counter()
for path in header_paths:
    try:
        ds = pydicom.dcmread(path, stop_before_pixels=True)
    except Exception:
        failed += 1
        continue
    for tag in ('PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'ImagePositionPatient', 'PixelSpacing'):
        present[tag] += bool(ds.get(tag))
    thickness[str(ds.get('SliceThickness', 'absent'))] += 1
header_rate = len(header_paths) / (time.perf_counter() - start)
report['header_read'] = {'per_second_one_process': round(header_rate, 1), 'failed': failed, 'tags_present_of_sample': dict(present), 'slice_thickness_top': thickness.most_common(8), 'estimated_full_scan_hours_one_process': round(n_slices / header_rate / 3600, 2), 'estimated_full_scan_hours_all_cpus': round(n_slices / header_rate / 3600 / max(NUM_WORKERS, 1), 2)}
print(report['header_read'])

## 4. Window and resize
Measures failed reads, and how large each stored image would be.

In [ ]:
def to_rgb(path, size):
    ds = pydicom.dcmread(path)
    hu = ds.pixel_array.astype('float32') * float(ds.get('RescaleSlope', 1)) + float(ds.get('RescaleIntercept', 0))
    channels = [np.clip((hu - (c - w / 2)) / w, 0, 1) for c, w in WINDOWS]
    image = (np.stack(channels, axis=-1) * 255).astype('uint8')
    return cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)

report['stored_size_kb'], pixel_failures = {}, 0
for size in IMAGE_SIZES:
    jpg, png = [], []
    for path in paths[:PIXEL_SAMPLE]:
        try:
            image = to_rgb(path, size)
        except Exception:
            pixel_failures += 1
            continue
        jpg.append(len(cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, 95])[1]))
        png.append(len(cv2.imencode('.png', image)[1]))
    report['stored_size_kb'][size] = {'jpeg95': round(float(np.mean(jpg)) / 1024, 1), 'png': round(float(np.mean(png)) / 1024, 1)}
report['pixel_read_failures'] = pixel_failures
print(report['stored_size_kb'], f'Failed reads: {pixel_failures} of {PIXEL_SAMPLE * len(IMAGE_SIZES)}')

## 5. Read data on the fly
Measures images per second through a DataLoader. This is usually the bottleneck with DICOM.

In [ ]:
class Slices(Dataset):
    def __init__(self, paths, targets, size):
        self.paths, self.targets, self.size = paths, targets, size
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        try:
            image = to_rgb(self.paths[i], self.size)
        except Exception:
            image = np.zeros((self.size, self.size, 3), 'uint8')
        return torch.from_numpy(image).permute(2, 0, 1).float() / 255, self.targets[i]

report['loader_images_per_second'] = {}
for size in IMAGE_SIZES:
    loader = DataLoader(Slices(paths[:LOADER_SAMPLE], targets[:LOADER_SAMPLE], size), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    start, seen = time.perf_counter(), 0
    for images, _ in loader:
        seen += len(images)
    report['loader_images_per_second'][size] = round(seen / (time.perf_counter() - start), 1)
print('Reading DICOM on the fly, images per second:', report['loader_images_per_second'])

## 6. Training speed
Uses random data, so this measures the GPU alone.

In [ ]:
def train_speed(name, size):
    model = getattr(torchvision.models, name)(weights=None, num_classes=1).to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler(device, enabled=device == 'cuda')
    images = torch.rand(BATCH_SIZE, 3, size, size, device=device)
    labels_ = torch.rand(BATCH_SIZE, 1, device=device).round()
    for step in range(TRAIN_STEPS + WARMUP_STEPS):
        if step == WARMUP_STEPS:
            start = time.perf_counter()
        optimiser.zero_grad(set_to_none=True)
        with torch.amp.autocast(device, enabled=device == 'cuda'):
            loss = torch.nn.functional.binary_cross_entropy_with_logits(model(images), labels_)
        scaler.scale(loss).backward()
        scaler.step(optimiser)
        scaler.update()
    if device == 'cuda':
        torch.cuda.synchronize()
    return round(BATCH_SIZE * TRAIN_STEPS / (time.perf_counter() - start), 1)

report['training_images_per_second'] = {}
for name in MODELS:
    for size in IMAGE_SIZES:
        try:
            speed = train_speed(name, size)
        except Exception as error:  # usually out of GPU memory at this batch size
            speed = f'failed: {str(error)[:80]}'
        report['training_images_per_second'][f'{name}@{size}'] = speed
        print(name, size, speed)
        if device == 'cuda':
            torch.cuda.empty_cache()

## 7. Summary
Estimates assume every labelled slice is used in each epoch. A class-balanced sampler can use fewer slices per epoch. Limits in the settings cell are unconfirmed, so compare them with your Kaggle account.

In [ ]:
rows = []
for key, train_rate in report['training_images_per_second'].items():
    size = int(key.split('@')[1])
    loader_rate = report['loader_images_per_second'][size]
    if isinstance(train_rate, str):
        rows.append({'run': key, 'limited_by': train_rate})
        continue
    rate = min(train_rate, loader_rate)
    hours = n_slices / rate / 3600
    rows.append({'run': key, 'limited_by': 'reading data' if loader_rate < train_rate else 'GPU', 'images_per_s': rate, 'hours_per_epoch': round(hours, 1), 'sessions_per_epoch': round(hours / SESSION_HOURS, 2), 'epochs_per_week': round(WEEKLY_GPU_HOURS / hours, 2)})
report['estimates'] = rows
report['preprocess_estimates'] = {size: {'jpeg95_output_gb': round(report['stored_size_kb'][size]['jpeg95'] * n_slices / 1024 ** 2, 1), 'png_output_gb': round(report['stored_size_kb'][size]['png'] * n_slices / 1024 ** 2, 1), 'hours': round(n_slices / report['loader_images_per_second'][size] / 3600, 1), 'fits_output_limit_as_jpeg': report['stored_size_kb'][size]['jpeg95'] * n_slices / 1024 ** 2 <= OUTPUT_LIMIT_GB} for size in IMAGE_SIZES}
report['disk_free_gb'] = round(shutil.disk_usage(WORK_DIR).free / 1e9, 1)

with open(os.path.join(WORK_DIR, 'rsna_timing_report.json'), 'w') as f:
    json.dump(report, f, indent=2)
print(pd.DataFrame(rows).to_string(index=False))
print()
print(json.dumps(report, indent=2))